# Testing Edge Cases

In [7]:
# Testing Diet and Allergy Case Profiles
#Importing essential libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from EVALUATION_METRICS import nutrient_contribution_test

In [ ]:
#Hard clinical constraints to introduce hybrid filtering 

#Vitamin D is not present in a lot of foods
#Recommended foods show up with near zero Vitamin D due to
#Vector Dot Product
#Before running the Cosine Similarity Calculation:
#Check for a patient's elevated need for Vitamin D
#If they do, the recommender should apply a filter to the food matrix
#To restrict the search space to find items that contain Vitamin D

def recommender_hard_constraints(patient_vector, food_db, top_n=10):

    #Nutrient order
    target_fibre = patient_vector[0]
    target_pufa = patient_vector[1]
    target_magnesium = patient_vector[2]
    target_vit_d = patient_vector[3]
    target_zinc = patient_vector[4]

    #Creating copies of databases for filtering
    filtered_food_db = food_db.copy()

    #Elevated need for Magnesium
    if target_fibre > 25:
        #Filtering out the foods that don't contain a lot of Zinc
        fibre_mask = filtered_food_db['Fibre, total dietary'] > 0.5
        filtered_food_db = filtered_food_db[fibre_mask]

    #Elevated need for Magnesium
    if target_pufa > 12:
        #Filtering out the foods that don't contain a lot of Zinc
        pufa_mask = filtered_food_db['Fatty acids, total polyunsaturated'] > 0.5
        filtered_food_db = filtered_food_db[pufa_mask]

    #Elevated need for Magnesium
    if target_magnesium > 320:
        #Filtering out the foods that don't contain a lot of Zinc
        magnesium_mask = filtered_food_db['Magnesium, Mg'] > 0.5
        filtered_food_db = filtered_food_db[magnesium_mask]

    #Elevated need for vitamin D
    if target_vit_d > 10:
        #Filtering out the foods that don't contain a lot of Vitamin D
        vit_d_mask = filtered_food_db['Vitamin_D_Total_UG'] > 1.0
        filtered_food_db = filtered_food_db[vit_d_mask]

    #Elevated need for Zinc
    if target_zinc > 7:
        #Filtering out the foods that don't contain a lot of Zinc
        zinc_mask = filtered_food_db['Zinc, Zn'] > 0.5
        filtered_food_db = filtered_food_db[zinc_mask]

    #If the filtering is too restrictive and returns nothing, reset to the full database
    if filtered_food_db.empty:
        filtered_food_db = food_db.copy()
    
    #Isolating numeric columns from the filtered database
    nutrients_5d_order = [
        'Fibre, total dietary', 
        'Fatty acids, total polyunsaturated',
        'Magnesium, Mg',
        'Vitamin_D_Total_UG',
        'Zinc, Zn'
    ]
    #Filtering
    food_numeric_matrix = filtered_food_db[nutrients_5d_order].values

    #Converting the patient vector to 2D and scaling it normally
    scaler = MinMaxScaler()
    scaled_food_db = scaler.fit_transform(food_numeric_matrix)
    vector_2d = np.array(patient_vector).reshape(1, -1)
    scaled_patient_vector = scaler.transform(vector_2d)

    #Calculating the similarity strictly across the subspace
    score_similarity = cosine_similarity(scaled_patient_vector, scaled_food_db)[0]
    
    #Creating count for the top 10 recommended foods

    #Attaching the scores back and treturning the top results
    results_df = filtered_food_db.copy()
    results_df['Match_Score (%)'] = np.round(score_similarity * 100, 2)
    return results_df.sort_values(by='Match_Score (%)', ascending=False).head(top_n)
        

In [ ]:
#-----------Vegan-----------

#Vegan PMOS Profile

#Loading 5d Food Matrix Database
food_matrix_5d = pd.read_csv("food_matrix_5d_categorised.csv")

#Defining the animal-based categories to completely exclude
animal_categories = [
    'Dairy and Egg Products',
    'Poultry Products',
    'Beef Products',
    'Finfish and Shellfish Products',
    'Pork Products',
    'Lamb, Veal, and Game Products',
    'Sausages and Luncheon Meats',
    'Restaurant Foods' #Could contain hidden animals/broths
]

#Applying the Vegan masking to the 5D Matrix (the one with the category names attached)
#The ~ symbol means "NOT in this list"
vegan_food_db = food_matrix_5d[~food_matrix_5d['category_name'].isin(animal_categories)].copy()

#Creating a sythentic test edge case patient vector
#High Vitamin D and High Zinc needs 
vegan_pmos_vector = [30.0, 15.0, 400.0, 25.0, 15.0]

#Running the recommender using the restricted Vegan database
print("Vegan Profile Stress Test: Top 10 Recommendations")
vegan_recs = recommender_hard_constraints(
    patient_vector=vegan_pmos_vector,
    food_db=vegan_food_db,
    top_n=10
)

#Display the foods and their categories to prove no animal products slipped through
print(vegan_recs[['food_description', 'category_name', 'Match_Score (%)']])

#Testing the clinical relevance of the Vegan Diet
#Creating a temporary dataframe with just this one synthetic patient to run the metric
mock_patient_df = pd.DataFrame([vegan_pmos_vector], columns=[
    'Fibre, total dietary', 
    'Fatty acids, total polyunsaturated',
    'Magnesium, Mg',
    'Vitamin_D_Total_UG',
    'Zinc, Zn'
])

#Need the raw math so a bypass vector function is created to return it
def bypass_vector(record):
    return record.values

#Printing the Nutrient Contribution of the Vegan diet
print("-----------------------------------------")
print("Vegan PMOS Profile: Nutrient Contribution")
vegan_relevance = nutrient_contribution_test(
    patient_df=mock_patient_df,
    food_db=food_matrix_5d,
    vector_function=bypass_vector,
    recommender_function=recommender_hard_constraints,
    top_n=10
)

#The recommender successfully filters out the foods and recommends vegan appropriate foods.
#However, a future improvement would be to diversify the recommendations as many of the recommendations are types of milk


Vegan Profile Stress Test: Top 10 Recommendations
                                  food_description  \
242                              Mushroom, maitake   
360     Soy milk, unsweetened, plain, shelf stable   
3    Almond milk, unsweetened, plain, shelf stable   
359       Soy milk, sweetened, plain, refrigerated   
2    Almond milk, unsweetened, plain, refrigerated   
260     Oat milk, unsweetened, plain, refrigerated   
237                                Mushroom, beech   
240                          Mushroom, king oyster   
243                               Mushroom, oyster   
247                        Mushrooms, white button   

                         category_name  Match_Score (%)  
242  Vegetables and Vegetable Products            98.93  
360        Legumes and Legume Products            97.37  
3                            Beverages            94.97  
359        Legumes and Legume Products            91.71  
2                            Beverages            91.61  
260    

In [ ]:
#-----------Tree Nut & Peanut Allergy Profile-----------

#Defining the nut allergy category
nut_category = ['Nut and Seed Products']

#Filtering out the specific category
allergy_food_db = food_matrix_5d[~food_matrix_5d['category_name'].isin(nut_category)].copy()

#Filtering out lingering peanuts from the Legume category using a text search
allergy_food_db = allergy_food_db[~allergy_food_db['food_description'].str.contains('peanut', case=False, na=False)]

#Creating a sythetic elevated nutrients needs PMOS patient vector
#High PUFAs (20.0g) and high Magnesium (450.0mg) content to force and stress test the recommender to find foods with these contents
allergy_pmos_vector = [25.0, 20.0, 450.0, 10.0, 7.0]

#Running the recommender using restricted Allergy database
print("Severe Nut Allergy Profile Stress Test: Top 10 Recommendations")
allergy_recs = recommender_hard_constraints(
    patient_vector=allergy_pmos_vector,
    food_db=allergy_food_db,
    top_n=10
)

#Displaying the foods to prove no nuts slipped through
print(allergy_recs[['food_description', 'category_name', 'Match_Score (%)']])

#Testing the nutrient contribution relevance of the allergy diet
mock_allergy_df = pd.DataFrame([allergy_pmos_vector], columns=[
    'Fiber, total dietary', 
    'Fatty acids, total polyunsaturated',
    'Magnesium, Mg',
    'Vitamin_D_Total_UG',
    'Zinc, Zn'
])

#Need the raw math so a bypass vector function is created to return it
def bypass_vector(record):
    return record.values

#Printing the Nutrient Contribution of the Vegan diet
nut_allergen_relevance = nutrient_contribution_test(
    patient_df=mock_allergy_df,
    food_db=allergy_food_db,
    vector_function=bypass_vector,
    recommender_function=recommender_hard_constraints,
    top_n=10
)

#The recommender recommends suitable foods for a profile with a severe nut allergy, such as legumes and carbohydrates.

Severe Nut Allergy Profile Stress Test: Top 10 Recommendations
                                      food_description  \
138                          Edamame, frozen, prepared   
74                 Bread, white, commercially prepared   
75           Bread, whole-wheat, commercially prepared   
329                   Restaurant, Latino, tamale, pork   
202                                 Hummus, commercial   
328  Restaurant, Latino, pupusas con frijoles (pupu...   
248                          Mustard, prepared, yellow   
125               Cookies, oatmeal, soft, with raisins   
325                  Refried beans, canned, vegetarian   
339   Sauce, pasta, spaghetti/marinara, ready-to-serve   

                         category_name  Match_Score (%)  
138  Vegetables and Vegetable Products            96.53  
74                      Baked Products            95.67  
75                      Baked Products            94.04  
329                   Restaurant Foods            93.61  
202     

In [11]:
#----Vegetarian----

#Defining the meat-based categories to completely exclude
meat_categories = [
    'Poultry Products',
    'Beef Products',
    'Finfish and Shellfish Products',
    'Pork Products',
    'Lamb, Veal, and Game Products',
    'Sausages and Luncheon Meats',
    'Restaurant Foods' #Could contain hidden animals/broths
]

#Applying the Vegetarian masking to the 5D Matrix (the one with the category names attached)
#The ~ symbol means "NOT in this list"
vegetarian_food_db = food_matrix_5d[~food_matrix_5d['category_name'].isin(meat_categories)].copy()

#Creating a sythentic test edge case patient vector
#High Vitamin D and High Zinc needs 
vegetarian_pmos_vector = [30.0, 15.0, 400.0, 25.0, 15.0]

#Running the recommender using the restricted Vegetarian database
print("Vegetarian Profile Stress Test: Top 10 Recommendations")
vegetarian_recs = recommender_hard_constraints(
    patient_vector=vegetarian_pmos_vector,
    food_db=vegetarian_food_db,
    top_n=10
)

#Display the foods and their categories to prove no animal products slipped through
print(vegetarian_recs[['food_description', 'category_name', 'Match_Score (%)']])

#Testing the clinical relevance of the Vegan Diet
#Creating a temporary dataframe with just this one synthetic patient to run the metric
veg_mock_patient_df = pd.DataFrame([vegetarian_pmos_vector], columns=[
    'Fiber, total dietary', 
    'Fatty acids, total polyunsaturated',
    'Magnesium, Mg',
    'Vitamin_D_Total_UG',
    'Zinc, Zn'
])

#Need the raw math so a bypass vector function is created to return it
def bypass_vector(record):
    return record.values

#Printing the Nutrient Contribution of the Vegan diet
print("-----------------------------------------")
print("vegetarian PMOS Profile: Nutrient Contribution")
vegetarian_relevance = nutrient_contribution_test(
    patient_df=veg_mock_patient_df,
    food_db=food_matrix_5d,
    vector_function=bypass_vector,
    recommender_function=recommender_hard_constraints,
    top_n=10
)



Vegetarian Profile Stress Test: Top 10 Recommendations
                                      food_description  \
396                  Yogurt, Greek, strawberry, nonfat   
242                                  Mushroom, maitake   
398                          Yogurt, plain, whole milk   
82                                 Buttermilk, low fat   
104  Cheese, pasteurized process cheese food or pro...   
235   Milk, whole, 3.25% milkfat, with added vitamin D   
233  Milk, nonfat, fluid, with added vitamin A and ...   
146                    Eggs, Grade A, Large, egg whole   
232  Milk, lowfat, fluid, 1% milkfat, with added vi...   
142               Egg, whole, raw, frozen, pasteurized   

                         category_name  Match_Score (%)  
396             Dairy and Egg Products            97.96  
242  Vegetables and Vegetable Products            95.62  
398             Dairy and Egg Products            95.03  
82              Dairy and Egg Products            94.44  
104             

In [12]:
#----Pescatarian----

#Defining the meat-based categories to completely exclude
pesc_categories = [
    'Poultry Products',
    'Beef Products',
    'Pork Products',
    'Lamb, Veal, and Game Products',
    'Sausages and Luncheon Meats',
    'Restaurant Foods' #Could contain hidden animals/broths
]

#Applying the Pescatarian masking to the 5D Matrix (the one with the category names attached)
#The ~ symbol means "NOT in this list"
pesc_food_db = food_matrix_5d[~food_matrix_5d['category_name'].isin(pesc_categories)].copy()

#Creating a sythentic test edge case patient vector
#High Vitamin D and High Zinc needs 
pesc_pmos_vector = [30.0, 15.0, 400.0, 25.0, 15.0]

#Running the recommender using the restricted pescatarian database
print("Pescatarian Profile Stress Test: Top 10 Recommendations")
pesc_recs = recommender_hard_constraints(
    patient_vector=pesc_pmos_vector,
    food_db=pesc_food_db,
    top_n=10
)

#Display the foods and their categories to prove no animal products slipped through
print(pesc_recs[['food_description', 'category_name', 'Match_Score (%)']])

#Testing the clinical relevance of the Vegan Diet
#Creating a temporary dataframe with just this one synthetic patient to run the metric
pesc_mock_patient_df = pd.DataFrame([pesc_pmos_vector], columns=[
    'Fiber, total dietary', 
    'Fatty acids, total polyunsaturated',
    'Magnesium, Mg',
    'Vitamin_D_Total_UG',
    'Zinc, Zn'
])

#Need the raw math so a bypass vector function is created to return it
def bypass_vector(record):
    return record.values

#Printing the Nutrient Contribution of the Vegan diet
print("-----------------------------------------")
print("Pescatarian PMOS Profile: Nutrient Contribution")
vegetarian_relevance = nutrient_contribution_test(
    patient_df=pesc_mock_patient_df,
    food_db=food_matrix_5d,
    vector_function=bypass_vector,
    recommender_function=recommender_hard_constraints,
    top_n=10
)



Pescatarian Profile Stress Test: Top 10 Recommendations
                                      food_description  \
396                  Yogurt, Greek, strawberry, nonfat   
242                                  Mushroom, maitake   
158  Fish, tuna, light, canned in water, drained so...   
398                          Yogurt, plain, whole milk   
82                                 Buttermilk, low fat   
104  Cheese, pasteurized process cheese food or pro...   
235   Milk, whole, 3.25% milkfat, with added vitamin D   
233  Milk, nonfat, fluid, with added vitamin A and ...   
146                    Eggs, Grade A, Large, egg whole   
232  Milk, lowfat, fluid, 1% milkfat, with added vi...   

                         category_name  Match_Score (%)  
396             Dairy and Egg Products            97.96  
242  Vegetables and Vegetable Products            95.62  
158     Finfish and Shellfish Products            95.07  
398             Dairy and Egg Products            95.03  
82             

In [ ]:
#----Halal Diet----

#Defining the meat-based categories to completely exclude
halal_categories = [
    'Pork Products',
    'Sausages and Luncheon Meats',
    'Restaurant Foods' #Could contain hidden animals/broths
]

#Applying the halal masking to the 5D Matrix (the one with the category names attached)
#The ~ symbol means "NOT in this list"
halal_food_db = food_matrix_5d[~food_matrix_5d['category_name'].isin(halal_categories)].copy()

#Creating a sythentic test edge case patient vector
#High Vitamin D and High Zinc needs 
halal_pmos_vector = [30.0, 15.0, 400.0, 25.0, 15.0]

#Running the recommender using the restricted halal database
print("Halal Profile Stress Test: Top 10 Recommendations")
halal_recs = recommender_hard_constraints(
    patient_vector=halal_pmos_vector,
    food_db=halal_food_db,
    top_n=10
)

#Display the foods and their categories to prove no animal products slipped through
print(halal_recs[['food_description', 'category_name', 'Match_Score (%)']])

#Testing the clinical relevance of the Vegan Diet
#Creating a temporary dataframe with just this one synthetic patient to run the metric
halal_mock_patient_df = pd.DataFrame([halal_pmos_vector], columns=[
    'Fiber, total dietary', 
    'Fatty acids, total polyunsaturated',
    'Magnesium, Mg',
    'Vitamin_D_Total_UG',
    'Zinc, Zn'
])

#Need the raw math so a bypass vector function is created to return it
def bypass_vector(record):
    return record.values

#Printing the Nutrient Contribution of the Vegan diet
print("-----------------------------------------")
print("Halal PMOS Profile: Nutrient Contribution")
halal_relevance = nutrient_contribution_test(
    patient_df=halal_mock_patient_df,
    food_db=food_matrix_5d,
    vector_function=bypass_vector,
    recommender_function=recommender_hard_constraints,
    top_n=10
)



Halal Profile Stress Test: Top 10 Recommendations
                                      food_description  \
396                  Yogurt, Greek, strawberry, nonfat   
242                                  Mushroom, maitake   
158  Fish, tuna, light, canned in water, drained so...   
398                          Yogurt, plain, whole milk   
82                                 Buttermilk, low fat   
104  Cheese, pasteurized process cheese food or pro...   
235   Milk, whole, 3.25% milkfat, with added vitamin D   
233  Milk, nonfat, fluid, with added vitamin A and ...   
146                    Eggs, Grade A, Large, egg whole   
232  Milk, lowfat, fluid, 1% milkfat, with added vi...   

                         category_name  Match_Score (%)  
396             Dairy and Egg Products            97.96  
242  Vegetables and Vegetable Products            95.62  
158     Finfish and Shellfish Products            95.07  
398             Dairy and Egg Products            95.03  
82              Dairy

: 